In [ ]:
#ok for svm and knn
!pip install numpy pandas scikit-learn qiskit qiskit-machine-learning


In [ ]:
# Run this cell FIRST — restart kernel after it finishes
!pip install qiskit-ibm-runtime qiskit-machine-learning qiskit-algorithms --quiet

In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv('/content/survey lung cancer.csv')

In [ ]:
dfen2 = df.copy()
dfen2['GENDER'] = dfen2['GENDER'].map({'F':0,'M':1})
dfen2['LUNG_CANCER'] = dfen2['LUNG_CANCER'].map({'NO':0,'YES':1})

In [ ]:
# ------------------------------------------
#  Install imblearn once (if not installed)
# ------------------------------------------
!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# ------------------------------------------
# 1️⃣  Verify dataset & identify target column
# ------------------------------------------
# You already have dfs
print("Original shape:", dfen2.shape)
print("Original class distribution:")
print(dfen2['LUNG_CANCER'].value_counts())

# ------------------------------------------
# 2️⃣  Separate features (X) and target (y)
# ------------------------------------------
X = dfen2.drop(columns=['LUNG_CANCER']).values
y = dfen2['LUNG_CANCER'].values

# ------------------------------------------
# 3️⃣  Apply SMOTE only on the minority class
# ------------------------------------------
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# ------------------------------------------
# 4️⃣  Rebuild into a balanced DataFrame
# ------------------------------------------
columns = dfen2.drop(columns=['LUNG_CANCER']).columns
dfs_smotes = pd.DataFrame(X_res, columns=columns)
dfs_smotes['LUNG_CANCER'] = y_res


# ------------------------------------------
# 5️⃣  Check the new class balance
# ------------------------------------------
print("\nAfter SMOTE:")
print(dfs_smotes['LUNG_CANCER'].value_counts())
print("New shape:", dfs_smotes.shape)

In [ ]:
# replace 'class' with your target/label column name
dff = dfs_smotes.groupby('LUNG_CANCER', group_keys=False).sample(n=25, random_state=42)

dff.info()

<h1>QPU submission

In [ ]:
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
from sklearn.svm import SVC

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityQuantumKernel  # used for statevector fallback only

# IBM Quantum Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService, Batch, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_algorithms.state_fidelities import ComputeUncompute

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# IBM QUANTUM CREDENTIALS  <-- FILL IN YOUR OWN VALUES
# ============================================================
IBM_API_KEY = "Qu7hKMsA9nP7jTWHkjIPWHql6zZ0UM3HCWMU1PzhAA2h"   # e.g. "abc123xyz..."
IBM_CRN     = "crn:v1:bluemix:public:quantum-computing:us-east:a/640910746940485c9f7ceeab65a83d7f:b2b6a055-cbd8-4056-bc3c-c4855b6d206e::"               # e.g. "crn:v1:bluemix:public:..."

# Backend preference order (first available with enough qubits wins)
# Updated to match your available backends: ibm_marrakesh, ibm_kingston, ibm_fez
PREFERRED_BACKENDS = [
    "ibm_kingston",    # 156q Heron r2 – newest, lowest error rates
    "ibm_fez",         # 156q Heron r1
    "ibm_marrakesh",   # 156q Heron r2
]

# ============================================================
# USER CONTROLS
# ============================================================
N_RUNS            = 1
TEST_SIZE         = 0.4
NUM_REPEATS       = 1
RANDOM_SEED_BASE  = 42
AUTO_SAVE_PATH    = "qsvm_ibm_hardware_results.csv"
TARGET            = "LUNG_CANCER"
SHOTS             = 1024          # shots per circuit on hardware
USE_REAL_HARDWARE = True          # set False to run locally with FakeSherbrooke

# Fixed feature sequence (0-based indexing)
FIXED_FEATURES_IDX = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

# ============================================================
# CONNECT TO IBM QUANTUM
# ============================================================
def get_backend():
    """
    Authenticate with IBM Quantum Cloud (using CRN for IBM Cloud accounts)
    and return the least-busy suitable backend.
    """
    print("Connecting to IBM Quantum...")

    # Save account – channel='ibm_cloud' is required when using a CRN
    QiskitRuntimeService.save_account(
        channel="ibm_cloud",
        token=IBM_API_KEY,
        instance=IBM_CRN,
        overwrite=True,
    )

    service = QiskitRuntimeService(
        channel="ibm_cloud",
        token=IBM_API_KEY,
        instance=IBM_CRN,
    )

    # Try preferred backends first
    available = [b.name for b in service.backends(operational=True, simulator=False)]
    print(f"Available real backends: {available}")

    chosen = None
    for name in PREFERRED_BACKENDS:
        if name in available:
            backend = service.backend(name)
            if backend.configuration().n_qubits >= len(FIXED_FEATURES_IDX):
                chosen = backend
                print(f"Selected backend: {name}  ({backend.configuration().n_qubits} qubits)")
                break

    if chosen is None:
        # Fallback: least-busy backend with enough qubits
        chosen = service.least_busy(
            operational=True,
            simulator=False,
            min_num_qubits=len(FIXED_FEATURES_IDX),
        )
        print(f"Fallback backend: {chosen.name}  ({chosen.configuration().n_qubits} qubits)")

    return service, chosen


# ============================================================
# LOAD DATA
# ============================================================
try:
    dfs_smotes = dff
except NameError:
    print("Creating synthetic dataset for demonstration...")
    np.random.seed(42)
    n_samples = 540
    n_features = 15
    X_pos = np.random.randn(n_samples // 2, n_features) + 1
    X_neg = np.random.randn(n_samples // 2, n_features) - 1
    X_demo = np.vstack([X_pos, X_neg])
    y_demo = np.array([1] * (n_samples // 2) + [0] * (n_samples // 2))
    columns = [f"feature_{i}" for i in range(n_features)]
    dfs_smotes = pd.DataFrame(X_demo, columns=columns)
    dfs_smotes[TARGET] = y_demo
    print(f"Synthetic dataset created: {dfs_smotes.shape}")

X_raw = dfs_smotes.iloc[:, FIXED_FEATURES_IDX].values.astype(float)
y     = dfs_smotes[TARGET].values.astype(int)
feature_names = dfs_smotes.columns[FIXED_FEATURES_IDX].tolist()
print(f"Using fixed features: {feature_names}")

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy":    accuracy_score(y_true, y_pred),
        "Precision":   precision_score(y_true, y_pred, zero_division=0),
        "Recall":      recall_score(y_true, y_pred, zero_division=0),
        "F1":          f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":     roc_auc_score(y_true, y_prob),
        "PR-AUC":      average_precision_score(y_true, y_prob),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa":       cohen_kappa_score(y_true, y_pred),
    }

# ============================================================
# CUSTOM FEATURE MAP  (Dense Angle Encoding)
# ============================================================
def create_dense_angle_feature_map(num_features, circuit_depth=1, n_reuploads=1):
    """
    Dense angle encoding: Rx(x) Ry(2x) Rz(0.5x) + linear CNOT entanglement.
    """
    x  = ParameterVector('x', num_features)
    qc = QuantumCircuit(num_features)

    for _ in range(circuit_depth):
        for _ in range(n_reuploads):
            for i in range(num_features):
                qc.rx(x[i],       i)
                qc.ry(2.0 * x[i], i)
                qc.rz(0.5 * x[i], i)
            for i in range(num_features - 1):
                qc.cx(i, i + 1)
    return qc

# ============================================================
# QUANTUM KERNEL  –  real hardware path (Batch mode for Open plan)
# ============================================================

def _bind_and_transpile(feature_map, x_vec, pm):
    """
    Bind parameter values into the feature map and return ISA circuits.
    We do this manually so FidelityQuantumKernel never gets a chance to
    re-transpile and re-introduce unsupported gates like sxdg.
    """
    circuits = []
    params = sorted(feature_map.parameters, key=lambda p: p.name)
    for x in x_vec:
        bound = feature_map.assign_parameters(dict(zip(params, x)))
        circuits.append(bound)
    return pm.run(circuits)

def _kernel_from_counts(result_list, shots):
    """
    Extract transition probability (|<0|circ|0>|^2) from sampler results.
    Each result corresponds to one (x_i, x_j) fidelity circuit.
    """
    values = []
    for res in result_list:
        # SamplerV2 result: res.data.meas.get_counts()
        try:
            counts = res.data.meas.get_counts()
        except AttributeError:
            # fallback for older result format
            counts = res.data[0].meas.get_counts()
        zero_key = "0" * list(counts.keys())[0].__len__()
        # find the all-zeros bitstring key (may be "000...0" or integer 0)
        zero_count = 0
        for k, v in counts.items():
            if set(str(k)) <= {"0"}:
                zero_count = v
                break
        values.append(zero_count / shots)
    return values

def _build_fidelity_circuits(fm, x_vec_a, x_vec_b, pm):
    """
    Build ComputeUncompute fidelity circuits for all pairs (a_i, b_j).
    Circuit = fm(a_i) + fm(b_j)^dagger, then measure all qubits.
    Returns list of ISA circuits and list of (i,j) index pairs.
    """
    from qiskit.circuit.library import StatePreparation
    params = sorted(fm.parameters, key=lambda p: p.name)
    n_qubits = fm.num_qubits
    pairs, circuits = [], []

    for i, a in enumerate(x_vec_a):
        for j, b in enumerate(x_vec_b):
            qc = QuantumCircuit(n_qubits)
            # bind feature map for a
            bound_a = fm.assign_parameters(dict(zip(params, a)))
            # bind feature map for b (adjoint)
            bound_b = fm.assign_parameters(dict(zip(params, b)))
            qc.compose(bound_a, inplace=True)
            qc.compose(bound_b.inverse(), inplace=True)
            qc.measure_all()
            pairs.append((i, j))
            circuits.append(qc)

    # Transpile ALL circuits in one batch call — guarantees consistent ISA
    isa_circuits = pm.run(circuits)

    # Final gate check
    heron_ok = {"cz","rz","sx","x","measure","reset","barrier","delay","snapshot"}
    for idx, circ in enumerate(isa_circuits):
        bad = {inst.operation.name for inst in circ.data} - heron_ok
        if bad:
            raise ValueError(f"Circuit {idx} still has unsupported gates after transpilation: {bad}")

    return isa_circuits, pairs

def build_hardware_kernel(X_train, X_test, backend, shots=SHOTS):
    """
    Manually build ComputeUncompute fidelity circuits, transpile them
    once to ISA, then submit via Batch+SamplerV2.

    This completely bypasses FidelityQuantumKernel's internal re-transpilation
    which was the source of sxdg errors on Heron QPUs.

    Returns
    -------
    kernel_train : (n_train, n_train) ndarray
    kernel_test  : (n_test,  n_train) ndarray
    """
    n_features  = X_train.shape[1]
    n_train     = X_train.shape[0]
    n_test      = X_test.shape[0]

    feature_map = create_dense_angle_feature_map(
        num_features=n_features, circuit_depth=1, n_reuploads=1
    )

    # Build pass manager — explicit basis_gates locks out sxdg for good
    heron_basis = ["cz", "rz", "sx", "x", "measure", "reset"]
    print(f"  Building ISA pass manager for {backend.name}…")
    pm = generate_preset_pass_manager(
        backend=backend,
        optimization_level=3,
        basis_gates=heron_basis,
    )

    # ---- Train kernel (symmetric: only upper triangle needed) ----
    print(f"  Building train fidelity circuits ({n_train}×{n_train})…")
    train_circuits, train_pairs = _build_fidelity_circuits(
        feature_map, X_train, X_train, pm
    )
    print(f"  {len(train_circuits)} circuits built, depth={train_circuits[0].depth()}")

    # ---- Test kernel ----
    print(f"  Building test fidelity circuits ({n_test}×{n_train})…")
    test_circuits, test_pairs = _build_fidelity_circuits(
        feature_map, X_test, X_train, pm
    )
    print(f"  {len(test_circuits)} circuits built")

    # ---- Submit all circuits in one Batch ----
    all_circuits = train_circuits + test_circuits
    print(f"  Submitting {len(all_circuits)} circuits to {backend.name} via Batch…")

    with Batch(backend=backend) as batch:
        sampler = Sampler(mode=batch)
        sampler.options.default_shots = shots

        t0 = time.time()
        job = sampler.run(all_circuits)
        print(f"  Job submitted. Waiting for results (queue + execution)…")
        result = job.result()
        print(f"  Results received in {time.time()-t0:.1f}s")

    # ---- Unpack results ----
    n_train_circs = len(train_circuits)
    train_results = [result[i] for i in range(n_train_circs)]
    test_results  = [result[i + n_train_circs] for i in range(len(test_circuits))]

    train_values  = _kernel_from_counts(train_results, shots)
    test_values   = _kernel_from_counts(test_results,  shots)

    # ---- Assemble kernel matrices ----
    kernel_train = np.zeros((n_train, n_train))
    for (i, j), v in zip(train_pairs, train_values):
        kernel_train[i, j] = v
        kernel_train[j, i] = v   # symmetric

    kernel_test = np.zeros((n_test, n_train))
    for (i, j), v in zip(test_pairs, test_values):
        kernel_test[i, j] = v

    return kernel_train, kernel_test

# ============================================================
# QUANTUM KERNEL  –  local / statevector fallback
# ============================================================
def build_statevector_kernel(X_train, X_test):
    """
    Pure-statevector simulation (no noise).  Used when
    USE_REAL_HARDWARE=False or as a quick sanity check.
    """
    n_features  = X_train.shape[1]
    feature_map = create_dense_angle_feature_map(
        num_features=n_features, circuit_depth=1, n_reuploads=1
    )
    qkernel = FidelityStatevectorKernel(feature_map=feature_map)
    kernel_train = qkernel.evaluate(x_vec=X_train)
    kernel_test  = qkernel.evaluate(x_vec=X_test, y_vec=X_train)
    return kernel_train, kernel_test

# ============================================================
# QSVM PREDICTION
# ============================================================
def predict_qiskit_svc(X_train, y_train, X_test, backend=None):
    if USE_REAL_HARDWARE and backend is not None:
        kernel_train, kernel_test = build_hardware_kernel(
            X_train, X_test, backend, shots=SHOTS
        )
    else:
        print("  [Simulation mode] Using statevector kernel…")
        kernel_train, kernel_test = build_statevector_kernel(X_train, X_test)

    clf = SVC(kernel='precomputed', probability=True)
    clf.fit(kernel_train, y_train)

    probs = clf.decision_function(kernel_test)
    probs = 1 / (1 + np.exp(-probs))   # sigmoid → [0, 1]
    return probs

# ============================================================
# CONNECT (once, outside the loop)
# ============================================================
if USE_REAL_HARDWARE:
    service, backend = get_backend()
    print(f"\nReady – jobs will run on: {backend.name}\n")
    print("=" * 60)
    print("IMPORTANT: IBM hardware queue wait varies 1–24 hrs.")
    print("Monitor job status at https://quantum.ibm.com/jobs")
    print("=" * 60 + "\n")
else:
    backend = None
    print("Running in SIMULATION mode (no IBM credentials needed).\n")

# ============================================================
# MAIN LOOP
# ============================================================
summary      = []
total_jobs   = NUM_REPEATS
completed_jobs = 0

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, NUM_REPEATS + 1):
    print(f"\n--- Split {split}/{NUM_REPEATS} ---")

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE * split,
    )

    # Imputation + Scaling
    imp = SimpleImputer(strategy="median")
    sc  = StandardScaler()
    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test  = sc.transform(imp.transform(X_test))

    print(f"  Train: {X_train.shape}  |  Test: {X_test.shape}")

    # ---------- QSVM ----------
    wall_start = time.time()
    y_prob     = predict_qiskit_svc(X_train, y_train, X_test, backend=backend)
    runtime    = time.time() - wall_start

    # ---------- Metrics ----------
    metrics       = compute_metrics(y_test, y_prob)
    completed_jobs += 1
    progress      = (completed_jobs / total_jobs) * 100

    print(
        f"\n[{progress:6.2f}%] QSVM_IBM_Hardware | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row = {
        "Split":       split,
        "Backend":     backend.name if backend else "statevector",
        "Shots":       SHOTS if USE_REAL_HARDWARE else 0,
        "Accuracy":    metrics["Accuracy"],
        "ROC_AUC":     metrics["ROC-AUC"],
        "F1":          metrics["F1"],
        "Precision":   metrics["Precision"],
        "Sensitivity": metrics["Sensitivity"],
        "Specificity": metrics["Specificity"],
        "Kappa":       metrics["Kappa"],
        "Runtime_sec": runtime,
    }
    summary.append(row)
    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)
    print(f"  Auto-saved → {AUTO_SAVE_PATH}")

# ============================================================
# FINAL SUMMARY
# ============================================================
summary_df = (
    pd.DataFrame(summary)
    .sort_values("Accuracy", ascending=False)
    .reset_index(drop=True)
)

print("\n===== FINAL SORTED RESULTS =====")
print(summary_df.to_string(index=False))
print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")
